In [ ]:
from rio_cogeo.cogeo import cog_validate
import rasterio
import os
import numpy as np
import xarray as xr
import h5py
import rioxarray
import glob
import geopandas as gpd
from shapely.geometry import Polygon

In [ ]:
input_folder = 'imerg/input/'
output_folder = 'imerg/output/'
os.makedirs(output_folder, exist_ok=True)

In [ ]:
# # Snippet to download the files

# import boto3
# import os

# # Create a session using your profile
# session = boto3.Session(profile_name='your-profile-name')

# # Create an S3 client
# s3 = session.client('s3')

# def download_s3_folder(bucket_name, s3_folder, local_dir):
#     paginator = s3.get_paginator('list_objects_v2')
#     for page in paginator.paginate(Bucket=bucket_name, Prefix=s3_folder):
#         for obj in page.get('Contents', []):
#             target = os.path.join(local_dir, os.path.relpath(obj['Key'], s3_folder))
#             if not os.path.exists(os.path.dirname(target)):
#                 os.makedirs(os.path.dirname(target))
#             if obj['Key'][-1] == '/':
#                 continue
#             s3.download_file(bucket_name, obj['Key'], target)
#             print(f"Downloaded {obj['Key']} to {target}")

# # Example usage
# bucket_name = 'veda-data-store-dev'
# s3_folder = 'cyclone/IMERG/input/'
# local_dir = input_folder

# download_s3_folder(bucket_name, s3_folder, local_dir)



In [ ]:
# Define AOI as a shapely Polygon
aoi_coords = [[-102.8148701375, 6.1943456775], [-13.3448605043, 6.1943456775], 
              [-13.3448605043, 49.6429910636], [-102.8148701375, 49.6429910636], 
              [-102.8148701375, 6.1943456775]]
aoi_polygon = Polygon(aoi_coords)
aoi_gdf = gpd.GeoDataFrame({'geometry': [aoi_polygon]}, crs="EPSG:4326")

In [ ]:
for file_name in glob.glob(f'{input_folder}/*/*.HDF5'):
    if file_name.endswith('.HDF5'):
        date = file_name.split("/")[-1].split(".")[4].split('-')[0]
        ts= file_name.split("/")[-1].split(".")[4].split('-')[-1][1:]
        formatted_datetime = f"{date[:4]}-{date[4:6]}-{date[6:]}T{ts[:2]}:{ts[2:4]}:{ts[4:]}Z"
        output_file = os.path.join(output_folder, f"IMERG_precipitation_{formatted_datetime}.tif")

        try:
            # Open the HDF5 file
            with h5py.File(file_name, 'r') as f:
                # Extract variables
                precipitation_data = f['Grid']['precipitation'][:]
                latitudes = f['Grid']['lat'][:]
                longitudes = f['Grid']['lon'][:]

            print(f"Original shape of precipitation data: {precipitation_data.shape}")

            # Process precipitation data
            precipitation_data = np.squeeze(precipitation_data).T  # Squeeze and transpose
            precipitation_data = precipitation_data.astype('float32')  # Ensure dtype
            precipitation_data = np.ma.masked_equal(precipitation_data, -9999)  # Mask no-data

            # Create xarray DataArray
            xds = xr.DataArray(
                precipitation_data,
                dims=["y", "x"],
                coords={"y": latitudes, "x": longitudes},
                name="precipitation"
            )

            # Adjust coordinates and flip latitude
            xds = xds.assign_coords(x=(((xds.x + 180) % 360) - 180)).sortby("x")
            xds = xds.isel(y=slice(None, None, -1))

            # Set nodata value using set_nodata
            xds = xds.where(xds != 0, -9999.0)
            xds = xds.rio.set_nodata(-9999.0 )
            xds.attrs['_FillValue'] = -9999.0


            # Set spatial dimensions and CRS
            xds.rio.set_spatial_dims("x", "y", inplace=True)
            xds.rio.write_crs("EPSG:4326", inplace=True)

            xds_clipped = xds.rio.clip(aoi_gdf.geometry, aoi_gdf.crs, drop=True)

            # Write to COG
            xds_clipped.rio.to_raster(
                output_file,
                driver="COG",
                dtype="float32",
                nodata=-9999.0,
                tags={"NODATA": "-9999.0"} 
            )
            print(f"COG file saved: {output_file}")

            # Validate the COG
            if cog_validate(output_file):
                print(f"COG validation successful for: {output_file}")
            else:
                print(f"COG validation failed for: {output_file}")

        except Exception as e:
            print(f"Error processing file {file_name}: {e}")